# MIA Visualization

In [1]:
import os
from pathlib import Path
from typing import Any

import ipywidgets as widgets
import plotly.express as px
from IPython.display import display
from plotly.subplots import make_subplots
import plotly.graph_objects as go

In [2]:
train_steps_dcr = [0.482, 0.508, 0.504, 0.519, 0.544, 0.605, 0.715, 0.773, 0.725, 0.717]
train_steps_wb = [0.133, 0.127, 0.167, 0.133, 0.187, 0.526, 0.550, 0.520, 0.530, 0.540]
train_steps_bb = [0.130, 0.143, 0.097, 0.117, 0.123, 0.220, 0.263, 0.280, 0.257, 0.223]
train_steps = [5e03, 1e04, 1.5e04, 2.5e04, 5e04, 1e05, 2e05, 3e05, 4e05, 5e05]

diffusion_steps_dcr = [0.703, 0.720, 0.712, 0.730, 0.708, 0.704, 0.711, 0.725, 0.706]
diffusion_steps_wb = [0.557, 0.566, 0.533, 0.543,  0.550, 0.587, 0.540, 0.550, 0.477]
diffusion_steps_bb = [0.053, 0.077, 0.067, 0.073, 0.120, 0.310, 0.290, 0.263, 0.240]
diffusion_steps = [10, 20, 50, 80, 100, 500, 1000, 2000, 3000]

synthetic_size = ["0.5x","1x", "2x", "5x"]
bb_1k = [0.390, 0.575, 0.780, 0.900]
bb_5k = [0.273, 0.363, 0.430, 0.663]
bb_10k = [0.230, 0.263, 0.330, 0.450]
bb_20k = [0.100, 0.130, 0.115, 0.095]
dcr_10k = [0.715, 0.724, 0.721,  0.720]

batch_size_dcr = [0.521, 0.561, 0.630, 0.701, 0.725, 0.730]
batch_size_wb = [0.247, 0.553, 0.563, 0.553, 0.550, 0.543]
batch_size_bb = [0.100, 0.143, 0.247, 0.287, 0.263, 0.267]
batch_size = [128, 256, 512, 2048, 4096, 8192]

train_size = [1e03, 5e03, 1e04, 1.5e04, 2e04]
train_size_wb = [0.956, 0.763, 0.550, 0.34, 0.155]
train_size_bb = [0.573, 0.363, 0.263, 0.185, 0.10]
train_size_dcr = [0.892, 0.755, 0.715, 0.5569, 0.507]
train_size_ideal_dcr = [1e03/(1e03+1e04), 5e03/(5e03+1e04), 0.50000, 1.5e04/(1e04+1.5e04), 2e04/(1e04+2e04)]

model_variation = ["Narrow<br>Shallow<br>Short", "Narrow<br>Shallow", "Narrow", "Default", "Wide<br>Deep", "Wide<br>Deep<br>Long"]
mia_wb = [0.113, 0.127, 0.153, 0.543, 0.563, 0.5467]
mia_bb = [0.077, 0.083, 0.110, 0.253, 0.210 , 0.25]
dcr = [0.513, 0.503, 0.502, 0.729, 0.7339, 0.8004]
dcr_ideal = [0.5, 0.5, 0.5, 0.5, 0.5, 0.5]

In [3]:
FIG_HEIGHT = 1000
FIG_WIDTH = 1000
FONT_SIZE = 38

## Utilities

In [4]:
def format_metric_name(metric_name: str) -> str:
    """
    Prettify diagram texts by replacing
    snake case with regular title case.
    """
    output_words = []
    for word in metric_name.split("_"):
        word = word.title() if word not in ["FPR", "TPR"] else word.upper()
        output_words.append(word)

    return " ".join(output_words)

## Plotting

In [5]:
repo_abs_path = Path(os.path.abspath("")).parent.parent

plots_dir = f"{repo_abs_path}/examples/visualizations/diabetes_tf_training"

In [6]:
def customize_figure_layout(gen_fig: Any) -> None:
    gen_fig.update_layout(
        height=FIG_HEIGHT,
        width=FIG_WIDTH,
        font_color="black",
        legend=dict(
            y=1.0,
            x=0.5,
            xanchor="center",
            yanchor="bottom",
            orientation="h",
            valign="top",
            title_text="",
            font=dict(size=FONT_SIZE-4),
            title_font_family="Helvetica",
        ),
        title_font_family="Helvetica",
        title_x=0.5,
        title_y=0.975,
        margin=dict(l=30, r=30, t=75, b=25),
        plot_bgcolor="white",
        font=dict(size=FONT_SIZE, family="Helvetica"),
    )


def save_and_display_figure(gen_fig: Any, template_name: str, plots_dir: str) -> None:
    plot_png_path = f"{plots_dir}/{template_name}.png"
    plot_pdf_path = f"{plots_dir}/{template_name}.pdf"

    os.makedirs(plots_dir, exist_ok=True)
    gen_fig.write_image(plot_png_path, scale=2)
    gen_fig.write_image(plot_pdf_path)

    display(gen_fig)

In [7]:
df = {"Train Steps": train_steps, "TF (WB)": train_steps_wb, "TF (BB)": train_steps_bb, "DCR": train_steps_dcr}

fig = px.line(
    data_frame=df,
    x="Train Steps",
    y=["TF (WB)", "TF (BB)", "DCR"],
    markers=True,
)
fig.add_hline(y=0.1, line_dash="dash", line_color="red", annotation_text="Ideal MIA", annotation_position="bottom right", line_width=6)
fig.add_hline(y=0.5, line_dash="dash", line_color="#660066", annotation_text="Ideal DCR", annotation_position="bottom right", line_width=6)
fig.data[0].line.color = "#53abff"
fig.data[1].line.color = "#f78d8d"
fig.data[2].line.color = "#660066"
fig.update_layout(
    xaxis=dict(
        tickmode='array',
        tickvals=[5e03, 1e04, 2e04, 5e04, 1e05, 2e05, 5e05],
    ),
)
fig.update_xaxes(
    type="log", ticks="outside", tickwidth=4, ticklen=7.5, linewidth=4, linecolor="black", tickangle=-25, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE, tickformat=".1E"
)
fig.update_yaxes(
    ticks="outside", tickwidth=4, ticklen=7.5, linewidth=4, linecolor="black", title = "Metric Score", tickangle=0, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE,
)
fig.update_traces(marker=dict(size=20), line=dict(width=8))

customize_figure_layout(fig)
save_and_display_figure(fig, "diabetes_tf_dcr_train_steps", plots_dir)

In [8]:
df = {"Diffusion Steps": diffusion_steps, "TF (WB)": diffusion_steps_wb, "TF (BB)": diffusion_steps_bb, "DCR": diffusion_steps_dcr}

fig = px.line(
    data_frame=df,
    x="Diffusion Steps",
    y=["TF (WB)", "TF (BB)", "DCR"],
    markers=True,
)
fig.data[0].line.color = "#53abff"
fig.data[1].line.color = "#f78d8d"
fig.data[2].line.color = "#660066"
fig.add_hline(y=0.1, line_dash="dash", line_color="red", annotation_text="Ideal MIA", annotation_position="bottom right", line_width=6)
fig.add_hline(y=0.5, line_dash="dash", line_color="#660066", annotation_text="Ideal DCR", annotation_position="bottom", line_width=6)
fig.update_layout(
    xaxis=dict(
        tickmode='array',
        tickvals=[1e01, 2e01, 5e01, 1e02, 2e02, 5e02, 1e03, 2e03, 4e03],
    ),
)
fig.update_xaxes(
    type="log", ticks="outside", tickwidth=4, ticklen=7.5, linewidth=4, linecolor="black", tickangle=-25, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE, tickformat=".1E"
)
fig.update_yaxes(
    ticks="outside", tickwidth=4, ticklen=7.5, linewidth=4, linecolor="black", title = "Metric Score", tickangle=0, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE,
)
fig.update_traces(marker=dict(size=20), line=dict(width=8))
fig.update_traces(marker=dict(size=20), line=dict(width=8))

customize_figure_layout(fig)
save_and_display_figure(fig, "diabetes_tf_dcr_diffusion_steps", plots_dir)

In [9]:
df = {"Synthetic Size": synthetic_size, "TF (BB) 1K": bb_1k, "TF (BB) 5K": bb_5k, "TF (BB) 10K": bb_10k, "TF (BB) 20K": bb_20k, "DCR 10K": dcr_10k}

fig = px.line(
    data_frame=df,
    x="Synthetic Size",
    y=["TF (BB) 1K", "TF (BB) 5K", "TF (BB) 10K", "TF (BB) 20K", "DCR 10K"],
    markers=True,
)
fig.data[4].line.color = "#660066"
fig.add_hline(y=0.1, line_dash="dash", line_color="red", annotation_text="Ideal MIA", annotation_position="bottom right", line_width=6)
fig.add_hline(y=0.5, line_dash="dash", line_color="#660066", annotation_text="Ideal DCR", annotation_position="bottom right", line_width=6)
fig.update_layout(
    xaxis=dict(
        tickmode='array',
        tickvals=["0.5x", "1x", "2x", "5x"],
    ),
)
fig.update_xaxes(
   ticks="outside", tickwidth=4, ticklen=7.5, linewidth=4, linecolor="black", tickangle=0, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE,
)
fig.update_yaxes(
    ticks="outside", tickwidth=4, ticklen=7.5, linewidth=4, linecolor="black", title = "Metric Score", tickangle=0, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE,
)
fig.update_traces(marker=dict(size=20), line=dict(width=8))

customize_figure_layout(fig)
save_and_display_figure(fig, "diabetes_tf_dcr_synthetic_size", plots_dir)

In [10]:
df = {"Batch Size": batch_size, "TF (WB)": batch_size_wb, "TF (BB)": batch_size_bb, "DCR": batch_size_dcr}

fig = px.line(
    data_frame=df,
    x="Batch Size",
    y=["TF (WB)", "TF (BB)", "DCR"],
    markers=True,
    log_x=True,

)
fig.data[0].line.color = "#53abff"
fig.data[1].line.color = "#f78d8d"
fig.data[2].line.color = "#660066"
fig.add_hline(y=0.1, line_dash="dash", line_color="red", annotation_text="Ideal MIA", annotation_position="bottom right", line_width=6)
fig.add_hline(y=0.5, line_dash="dash", line_color="#660066", annotation_text="Ideal DCR", annotation_position="bottom right", line_width=6)
fig.update_layout(
    xaxis=dict(
        tickmode='array',
        tickvals=["128", "256", "512","2048", "4096", "8192"],
    ),
)
fig.update_xaxes(
    ticks="outside", tickwidth=4, ticklen=7.5, linewidth=4, linecolor="black", tickangle=0, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE,
)
fig.update_yaxes(
    ticks="outside", tickwidth=4, ticklen=7.5, linewidth=4, linecolor="black", title = "Metric Score", tickangle=0, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE,
)
fig.update_traces(marker=dict(size=20), line=dict(width=8))

customize_figure_layout(fig)
save_and_display_figure(fig, "diabetes_tf_dcr_batch_size", plots_dir)

In [11]:
df = {"Train Size": train_size, "TF (WB)": train_size_wb, "TF (BB)": train_size_bb, "DCR": train_size_dcr, "Ideal DCR": train_size_ideal_dcr}

fig = px.line(
    data_frame=df,
    x="Train Size",
    y=["TF (WB)", "TF (BB)", "DCR"],
    markers=True,

)
fig.add_trace(go.Scatter(x=df["Train Size"], y=df["Ideal DCR"], mode='lines', name='Ideal DCR'))
fig.data[0].line.color = "#53abff"
fig.data[1].line.color = "#f78d8d"
fig.data[2].line.color = "#660066"
fig.data[3].line.color = "#660066"
fig.data[3].line.dash = "dash"
fig.add_hline(y=0.1, line_dash="dash", line_color="red", annotation_text="Ideal MIA", annotation_position="bottom right", line_width=6)
fig.update_layout(
    xaxis=dict(
        tickmode='array',
        tickvals=[1e3, 5e3, 1e4, 2e4],
    ),
)
fig.update_xaxes(
    type="log", ticks="outside", tickwidth=4, ticklen=7.5, linewidth=4, linecolor="black", tickangle=-25, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE, tickformat=".1E",
)
fig.update_yaxes(
    ticks="outside", tickwidth=4, ticklen=7.5, linewidth=4, linecolor="black", title = "Metric Score", tickangle=0, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE,
)
fig.update_traces(marker=dict(size=20), line=dict(width=8))

customize_figure_layout(fig)
save_and_display_figure(fig, "diabetes_tf_dcr_train_size", plots_dir)

In [12]:
df = {"Model Variations": model_variation, "TF (WB)": mia_wb, "TF (BB)": mia_bb, "DCR": dcr}

fig = px.line(
    data_frame=df,
    x="Model Variations",
    y=["TF (WB)", "TF (BB)", "DCR"],
    markers=True,

)
fig.data[0].line.color = "#53abff"
fig.data[1].line.color = "#f78d8d"
fig.data[2].line.color = "#660066"
fig.add_hline(y=0.1, line_dash="dash", line_color="red", annotation_text="Ideal MIA", annotation_position="bottom right", line_width=6)
fig.add_hline(y=1/2, line_dash="dash", line_color="#660066", annotation_text="Ideal DCR", annotation_position="bottom right", line_width=6)
fig.update_layout(
    xaxis=dict(
        tickmode='array',
        tickvals=model_variation,
    ),
)
fig.update_xaxes(
   ticks="outside", tickwidth=4, ticklen=7.5, linewidth=4, linecolor="black", tickangle=0, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE,
)
fig.update_yaxes(
    ticks="outside", tickwidth=4, ticklen=7.5, linewidth=4, linecolor="black", title = "Metric Score", tickangle=0, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE,
)
fig.update_traces(marker=dict(size=20), line=dict(width=8))

customize_figure_layout(fig)
save_and_display_figure(fig, "diabetes_tf_dcr_model_variation", plots_dir)